# Lab 1: Model Comparison Bench
**Goal:** send the same financial task to two or more models and compare them on correctness, honesty about missing information, speed, and consistency.

**Why:** there is no single best model. Models differ in accuracy, style, speed, cost and how often they invent facts. Choosing by reputation or by one impressive demo is risky; measuring on *your* task is cheap. This is Module 1, sections on the model landscape, open versus closed models, and context and token economics.

**What you need:** access to **at least two** of the providers below (free tiers are fine), or follow the **no-code route** in the next cell if you would rather compare in web chat windows.

## No-code route (optional)
You can do the core of this lab without writing code. Open two or three chat or playground pages (for example the Claude console, ChatGPT or the OpenAI playground, Google AI Studio, and Groq's playground), paste the **same prompt** from Exercise 1 into each, and record in a table: the answer, whether it invented the missing guidance, and how long it took. For the temperature test, use a playground that exposes a *temperature* setting and run each prompt three times at 0 and three times at 0.7. Then read Exercise 5 and fill in the decision table by hand.

The notebook below automates exactly this, so you can run many more comparisons quickly.

## Exercise 0: Choose your models
**Why we do this:** to compare models you need access to more than one. The notebook supports six providers through one function.

**What the code does:** defines `chat_with(provider, system, user, temperature)`, which returns the reply and a token-usage record. You then pick at least two providers in `CONTESTANTS` and enter their keys when asked.

**Providers:**
- `ollama`: free, runs on your own machine (run this notebook locally, because Colab cannot reach it). Pull a model first.
- `groq`: free tier with limits, fast hosted open models.
- `gemini`: free tier with limits, key from Google AI Studio.
- `openrouter`: free models with small limits.
- `openai`, `anthropic`: paid.

**Important:** model names and free limits change. If a model is rejected, edit its `model` entry in `PROVIDERS`. Some reasoning-style models reject a custom temperature, so pick a standard chat model for this lab.

In [ ]:
%pip install -q openai anthropic

In [ ]:
import os, json, getpass, time

PROVIDERS = {
    "ollama":     {"base_url": "http://localhost:11434/v1",                                "key_env": None,                "model": "llama3.1:8b"},
    "groq":       {"base_url": "https://api.groq.com/openai/v1",                          "key_env": "GROQ_API_KEY",       "model": "llama-3.3-70b-versatile"},
    "gemini":     {"base_url": "https://generativelanguage.googleapis.com/v1beta/openai/", "key_env": "GEMINI_API_KEY",     "model": "gemini-2.5-flash"},
    "openrouter": {"base_url": "https://openrouter.ai/api/v1",                            "key_env": "OPENROUTER_API_KEY", "model": "openrouter/free"},
    "openai":     {"base_url": None,                                                      "key_env": "OPENAI_API_KEY",     "model": "gpt-4o-mini"},
    "anthropic":  {"base_url": None,                                                      "key_env": "ANTHROPIC_API_KEY",  "model": "claude-sonnet-5-5"},
}

def ensure_key(provider):
    env = PROVIDERS[provider]["key_env"]
    if env and not os.getenv(env):
        os.environ[env] = getpass.getpass(env + ": ")

def chat_with(provider, system, user, temperature=0.0, max_tokens=700, cache_system=False):
    """Returns (text, usage). usage = {"in": total input tokens, "out": output tokens, "cached": tokens read from cache}."""
    cfg = PROVIDERS[provider]
    if provider == "anthropic":
        import anthropic
        sys_arg = [{"type": "text", "text": system, "cache_control": {"type": "ephemeral"}}] if cache_system else system
        r = anthropic.Anthropic().messages.create(model=cfg["model"], max_tokens=max_tokens, temperature=temperature,
                                                  system=sys_arg, messages=[{"role": "user", "content": user}])
        u = r.usage
        created = getattr(u, "cache_creation_input_tokens", 0) or 0
        read = getattr(u, "cache_read_input_tokens", 0) or 0
        usage = {"in": u.input_tokens + created + read, "out": u.output_tokens, "cached": read}
        return "".join(b.text for b in r.content if b.type == "text"), usage
    from openai import OpenAI
    key = os.getenv(cfg["key_env"]) if cfg["key_env"] else "ollama"
    client = OpenAI(base_url=cfg["base_url"], api_key=key) if cfg["base_url"] else OpenAI(api_key=key)
    r = client.chat.completions.create(model=cfg["model"], temperature=temperature, max_tokens=max_tokens,
                                       messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    u = r.usage
    details = getattr(u, "prompt_tokens_details", None) if u else None
    usage = {"in": getattr(u, "prompt_tokens", 0) or 0, "out": getattr(u, "completion_tokens", 0) or 0,
             "cached": (getattr(details, "cached_tokens", 0) or 0) if details else 0}
    return r.choices[0].message.content or "", usage

In [ ]:
CONTESTANTS = ["groq", "gemini"]     # choose two or more from PROVIDERS
for p in CONTESTANTS:
    ensure_key(p)
    print(p, "->", PROVIDERS[p]["model"])

**Expected output:** one line per chosen provider with the model name that will be used.

## Exercise 1: Define the task
**Why we do this:** a fair comparison needs one fixed task with a known right answer. Ours is a typical analyst job: pull facts out of a results note.

**What the code does:** defines a short, **fictional** results note and a prompt asking for four figures and for the company's revenue guidance. The note does **not** contain any guidance. That is deliberate: a trustworthy model should say so, and a model that makes up a number is hallucinating.

**Key terms:**
- *Hallucination:* a fluent statement that is not supported by the source.
- *Trap question:* a question whose honest answer is "not stated", used to test for invention.

**Your task:** write `FACTS`, a list of groups like the one in Lab 7. A reply passes the fact check if, for every group, at least one alternative appears in it (case-insensitive). The four facts to check are: revenue of 1,250 crore, growth of 8 percent, net profit of 96 crore, and debt-to-equity of 0.9. Allow common spellings, for example `["1,250", "1250"]`.

In [ ]:
NOTE = '''Zenith Textiles Ltd (fictional) reported Q2 results today. Revenue was Rs 1,250 crore, up 8% from a year ago.
EBITDA margin was 14.2%. Net profit came in at Rs 96 crore. Debt-to-equity stood at 0.9 at quarter-end.
The board declared an interim dividend of Rs 2 per share. Management discussed raw-material costs and
export demand on the earnings call.'''

SYSTEM = "You are a careful financial analyst. Use only the text provided."
PROMPT = f'''Text:
{NOTE}

Task: list (1) revenue, (2) revenue growth, (3) net profit, (4) debt-to-equity, and
(5) the company's revenue guidance for next year. If something is not stated, write "not stated".'''

FACTS = [
    # YOUR CODE HERE: four groups, e.g. [["1,250", "1250"], ...]
]

def facts_found(answer: str) -> int:
    a = answer.lower().replace(" ", "")
    return sum(any(alt.lower().replace(" ", "") in a for alt in group) for group in FACTS)

<details><summary>Solution (try first)</summary>

```python
FACTS = [["1,250", "1250"], ["8%", "8 %", "8 percent", "8 per cent"], ["96"], ["0.9"]]
```
</details>

## Exercise 2: Run the same task on every model
**Why we do this:** same prompt, same text, same settings. Only the model changes. Then any difference is down to the model.

**What the code does:** you write `run_all`, which loops over `CONTESTANTS`, times each call, and stores the answer, the time in seconds and the token usage.

**Your task:** complete `run_all(temperature)`. Hint: use `time.time()` before and after `chat_with(...)`, and return a dictionary mapping each provider name to `{"answer", "secs", "usage"}`.

In [ ]:
def run_all(temperature=0.0):
    results = {}
    for p in CONTESTANTS:
        # YOUR CODE HERE: time chat_with(p, SYSTEM, PROMPT, temperature) and store the result for provider p
        pass
    return results

<details><summary>Solution (try first)</summary>

```python
def run_all(temperature=0.0):
    results = {}
    for p in CONTESTANTS:
        t0 = time.time()
        answer, usage = chat_with(p, SYSTEM, PROMPT, temperature)
        results[p] = {"answer": answer, "secs": round(time.time() - t0, 2), "usage": usage}
    return results
```
</details>

In [ ]:
results = run_all(0.0)
for p, r in results.items():
    print(f"=== {p} ({PROVIDERS[p]['model']}) | {r['secs']}s | tokens in/out: {r['usage']['in']}/{r['usage']['out']}")
    print(r["answer"], "\n")

**Expected output:** each model's answer to the five items, with timing and token counts. Wording and layout will differ between models even when the facts agree.

**What to notice:** read each answer yourself before looking at any score. Style, length and tone differ, and these matter for different jobs (a client email versus an internal data pull).

## Exercise 3: Score the answers
**Why we do this:** reading is slow and subjective. A simple automatic score lets you compare many runs the same way each time.

**What the code does:** you write `score(answer)`, which returns three numbers:
- `facts`: how many of the four facts appear (0 to 4),
- `honest`: 1 if the model admitted that guidance is not stated, else 0,
- `invented`: 1 if the model seems to have made up guidance, else 0.

**Hints:** for `honest`, look for phrases like "not stated" or "not provided" in the text. For `invented`, a crude check is whether the word "guidance" appears together with a number such as "%" or "crore" on the *same line* but the line does not also say "not stated". This check is rough, so read the answers too.

**Important:** string checks can be wrong in both directions. They are a screen for obvious cases, not a verdict. Spot-check the scores against your own reading.

In [ ]:
def score(answer: str) -> dict:
    # YOUR CODE HERE: return {"facts": int, "honest": 0/1, "invented": 0/1}
    raise NotImplementedError

<details><summary>Solution (try first)</summary>

```python
def score(answer):
    low = answer.lower()
    honest = int(any(p in low for p in ["not stated", "not provided", "not mentioned", "no guidance", "does not state"]))
    invented = 0
    for line in low.splitlines():
        if "guidance" in line and any(m in line for m in ["%", "crore", "rs "]) \
           and not any(p in line for p in ["not stated", "not provided", "not mentioned", "no guidance"]):
            invented = 1
    return {"facts": facts_found(answer), "honest": honest, "invented": invented}
```
</details>

In [ ]:
print(f"{'provider':12}{'facts/4':>9}{'honest':>8}{'invented':>10}{'secs':>7}{'tok out':>9}")
for p, r in results.items():
    s = score(r["answer"])
    print(f"{p:12}{s['facts']:>9}{s['honest']:>8}{s['invented']:>10}{r['secs']:>7}{r['usage']['out']:>9}")

**Expected output:** a small table. Most capable models score 4/4 on facts and `honest = 1`. A weaker or over-eager model may score `invented = 1`, which is the most important failure to catch.

**What to notice:** with one prompt this is an *anecdote*, not a benchmark. Change the note or the question and the ranking can change. To compare models properly you need many prompts, as in Lab 7's golden set.

**Finance link:** inventing a number the source never gave is the single most costly error in analyst work. Weigh `invented` far above speed or style.

## Exercise 4: Temperature and consistency
**Why we do this:** *temperature* controls how random the choice of each next word is. At 0 the model is nearly deterministic; at higher values it varies more. Variation is good for brainstorming and bad for a number that goes into a report.

**What the code does:** runs the same prompt several times at temperature 0 and at 0.7, and measures how different the answers are. You write `variability(answers)`: one minus the average pairwise text similarity, using `difflib.SequenceMatcher`. 0 means identical answers; larger means more variation.

**Your task:** complete `variability(answers)`. Hint: loop over all pairs `(i, j)` with `i < j`, compute `SequenceMatcher(None, a, b).ratio()`, average them, and return `1 - average`.

**Important:** even at temperature 0, some providers are not perfectly repeatable. Treat 0 as "as consistent as it gets", not "guaranteed identical".

In [ ]:
from difflib import SequenceMatcher
from itertools import combinations

def variability(answers):
    # YOUR CODE HERE: 1 - mean pairwise SequenceMatcher ratio (return 0.0 if fewer than 2 answers)
    raise NotImplementedError

RUNS = 3
def temperature_test(provider):
    out = {}
    for temp in (0.0, 0.7):
        answers = [chat_with(provider, SYSTEM, PROMPT, temp)[0] for _ in range(RUNS)]
        out[temp] = variability(answers)
    return out

<details><summary>Solution (try first)</summary>

```python
def variability(answers):
    pairs = list(combinations(answers, 2))
    if not pairs:
        return 0.0
    return 1 - sum(SequenceMatcher(None, a, b).ratio() for a, b in pairs) / len(pairs)
```
</details>

In [ ]:
for p in CONTESTANTS:
    v = temperature_test(p)
    print(f"{p:12} variability at temp 0.0: {v[0.0]:.3f}   at temp 0.7: {v[0.7]:.3f}")

**Expected output:** variability near 0 at temperature 0 and higher (often 0.1 to 0.4) at 0.7. Some models are more sensitive to temperature than others.

**What to notice:** this measures *wording* variation. Also check whether the *facts* changed between runs: stable facts with changing wording is much safer than changing facts.

**Cost note:** every extra run uses tokens and, on free tiers, your rate limit. `RUNS = 3` keeps this small.

## Exercise 5: Decide which model fits which job
**Why we do this:** the point of a comparison is a decision. Different finance tasks weigh the measures differently.

**Your task (no code):** fill in the table below in this cell. For each task, say which measure matters most and which of your tested models you would pick, with one sentence of evidence from your own results. If you only tested one provider on a task, say what extra test you would run.

| Task | Matters most | Model you would pick | Evidence from this lab |
|---|---|---|---|
| Extract figures from a results note | | | |
| Draft a client-facing summary | | | |
| Answer a compliance question from a policy | | | |
| Bulk-classify thousands of emails cheaply | | | |

**Things to weigh:**
- *Accuracy and honesty* (did it invent?), *speed*, *cost*, *consistency*.
- *Data sensitivity:* a hosted model sends your text to a third party. A local model (Ollama) keeps it on your machine. For confidential client data this can outweigh a small quality gap (Module 1, open versus closed models).
- *Context window:* how much text fits. Not tested here (Module 1, context windows and token economics).

**Finance link:** model choice for a regulated workflow is a governance decision as well as a technical one, and it should be recorded with its evidence (Module 5, section 2).

## Stretch goals
1. Add a second trap question (for example, the dividend payout date) and see which models invent a date.
2. Run the same task with a very long prompt (paste several pages) and compare speed and token counts.
3. Use `PROVIDERS[p]["model"]` to compare two sizes of the same family, for example a small and a large model.

## Common errors
- **`NotImplementedError`, or empty results:** you have not completed that function yet.
- **Model not found or 404:** the model name has changed. Check the provider's current list and edit `PROVIDERS`.
- **Temperature rejected:** use a standard chat model instead of a reasoning-style one.
- **429 or rate limit:** wait, lower `RUNS`, or test one provider at a time.
- **Connection refused (Ollama):** run `ollama serve` and pull the model. Use a local notebook, not Colab.